# Data Cleaning and Feature Engineering
Preparing the PaySim dataset for machine learning by filtering data, removing unnecessary columns, and creating meaningful features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the dataset and sample 500,000 rows
file_path = '../data/PS_20174392719_1491204439457_log.csv'
df = pd.read_csv(file_path)
df = df.sample(n=500000, random_state=42)
print(f"Initial shape: {df.shape}")

## 1. Filter Transaction Types
In the PaySim dataset, fraudulent transactions only occur in `TRANSFER` and `CASH_OUT` transaction types. We keep only these to help the model learn more effectively.

In [ ]:
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
fraud_rate = df['isFraud'].mean() * 100
print(f"Shape after filtering: {df.shape}")
print(f"Fraud rate after filtering: {fraud_rate:.4f}%")

## 2. Drop Unnecessary Columns
- **nameOrig / nameDest**: These are unique IDs for the sender and receiver. There are too many unique values for a model to learn from, and they don't generalize to new customers.
- **isFlaggedFraud**: This is a simplistic rule-based flag from the original system (flags amounts > 200,000). It is highly inaccurate and doesn't provide useful information for a machine learning model.

In [ ]:
cols_to_drop = ['nameOrig', 'nameDest', 'isFlaggedFraud']
df = df.drop(columns=cols_to_drop)
print("Dropped columns: ", cols_to_drop)

## 3. Feature Engineering
- **errorBalanceOrig / errorBalanceDest**: These calculate the discrepancy between expected balances and actual balances. Fraudsters often manipulate accounts, leaving inconsistencies in the system. High errors strongly indicate suspicious activity.
- **hour**: The `step` variable represents hours passed since the start. By taking `step % 24`, we extract the time of day. Fraud might have temporal patterns (e.g., happening more often at night).
- **is_transfer**: Since we only have TRANSFER and CASH_OUT left, converting this to a binary 1/0 feature makes it easily digestible for machine learning algorithms.

In [ ]:
# Balance errors
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# Time of day
df['hour'] = df['step'] % 24

# Convert type to binary
df['is_transfer'] = (df['type'] == 'TRANSFER').astype(int)
df = df.drop(columns=['type'])

print("Engineered new features.")

## 4. Compare Features for Fraud vs Non-Fraud

In [ ]:
features_to_compare = ['errorBalanceOrig', 'errorBalanceDest', 'hour', 'is_transfer']
print("Average values for Non-Fraud (0) vs Fraud (1):\n")
print(df.groupby('isFraud')[features_to_compare].mean())

## 5. Plotting Balance Errors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Filter data for fraud and non-fraud
normal = df[df['isFraud'] == 0]
fraud = df[df['isFraud'] == 1]

# 1. Histogram for errorBalanceOrig
p1_orig = df['errorBalanceOrig'].quantile(0.01)
p99_orig = df['errorBalanceOrig'].quantile(0.99)

axes[0].hist(normal['errorBalanceOrig'], bins=50, range=(p1_orig, p99_orig), alpha=0.5, density=True, color='blue', label='Non-Fraud')
axes[0].hist(fraud['errorBalanceOrig'], bins=50, range=(p1_orig, p99_orig), alpha=0.5, density=True, color='red', label='Fraud')
axes[0].set_title('Density of errorBalanceOrig (1st-99th Pct)')
axes[0].set_xlabel('Error in Origin Balance')
axes[0].set_ylabel('Density')
axes[0].legend()

# 2. Histogram for errorBalanceDest
p1_dest = df['errorBalanceDest'].quantile(0.01)
p99_dest = df['errorBalanceDest'].quantile(0.99)

axes[1].hist(normal['errorBalanceDest'], bins=50, range=(p1_dest, p99_dest), alpha=0.5, density=True, color='blue', label='Non-Fraud')
axes[1].hist(fraud['errorBalanceDest'], bins=50, range=(p1_dest, p99_dest), alpha=0.5, density=True, color='red', label='Fraud')
axes[1].set_title('Density of errorBalanceDest (1st-99th Pct)')
axes[1].set_xlabel('Error in Destination Balance')
axes[1].set_ylabel('Density')
axes[1].legend()

# 3. Bar chart: % of transactions with abs(errorBalanceOrig) < 1
normal_zero_error_pct = (normal['errorBalanceOrig'].abs() < 1).mean() * 100
fraud_zero_error_pct = (fraud['errorBalanceOrig'].abs() < 1).mean() * 100

axes[2].bar(['Non-Fraud', 'Fraud'], [normal_zero_error_pct, fraud_zero_error_pct], color=['blue', 'red'], alpha=0.7)
axes[2].set_title('% of Transactions with Zero Origin Error')
axes[2].set_ylabel('Percentage (%)')

plt.tight_layout()
plt.show()

## 6. Save Processed Data

In [ ]:
df.to_csv('../data/processed.csv', index=False)
print("Saved final dataset to ../data/processed.csv")